# Example 03 -- Run your own RTL through the chipathon workshop padring

This notebook is the shortest path between "I have my own Verilog"
and "I have a fab-ready GDS that respects the shuttle padring." It
consumes the pre-built **workshop slot** from
https://github.com/Mauricio-xx/chipathon-2026-gf180mcu-padring (a fork
of the [wafer-space gf180mcu-project-template](https://github.com/wafer-space/gf180mcu-project-template)
that adds a 2935 x 2935 um slot mirroring
[JuanMoya/padring_gf180](https://github.com/JuanMoya/padring_gf180)).

The notebook will:

1. Stage the in-repo workshop padring (vendored at `../../resources/Integration/workshop_padring_librelane/`) into `~/eda/designs/chipathon_padring/template` (behind `RUN_STAGE_TEMPLATE`).
2. Clone the wafer-space GF180 PDK fork next to it (behind `RUN_CLONE_PDK`).
3. Let you write your `chip_core.sv` (behind `RUN_WRITE_CORE`).
4. Run the full `SLOT=workshop make librelane` flow (behind `RUN_FLOW`).

All `RUN_*` flags default to `False` so the first pass only prints
the commands -- flip each one when you are ready to commit.

**What the padring gives you (do not modify).** The `chip_top.sv` of
the template is fixed and wires the padring to your core via these
ports (parameters come from the `SLOT_WORKSHOP` block in
`src/slot_defines.svh`):

| port                          | width | direction | notes |
|-------------------------------|-------|-----------|-------|
| `clk`                         | 1     | input     | from the Schmitt-trigger clk_pad |
| `rst_n`                       | 1     | input     | active-low reset (from rst_n_pad) |
| `input_in`                    | 1     | input     | spare CMOS input (inputs[0].pad) |
| `input_pu`, `input_pd`        | 1 ea  | output    | tie to `'0`; no pull-ups/downs |
| `bidir_in[19:0]`              | 20    | input     | from the 20 config pads (bi_24t) |
| `bidir_out[19:0]`             | 20    | output    | drives the 20 config pads |
| `bidir_oe/cs/sl/ie/pu/pd[19:0]` | 20 each | output | pad-cell controls; tied defaults work |
| `analog[59:0]`                | 60    | inout     | 60 analog pads (asig_5p0, 5 V) |

**Your contract:**
- Produce a `module chip_core (...)` with the signature shown above.
- Drive `bidir_out` with whatever you want visible on the 20
  digital output pins.
- Either tie off or use `analog[]` in your custom analog IP; the
  digital flow leaves them as pure pass-through to the pad cells.

Default-safe controls for the pads (copy into every `chip_core`):

```verilog
assign input_pu  = '0;
assign input_pd  = '0;
assign bidir_oe  = '1;    // bidir pads drive outwards
assign bidir_cs  = '0;    // CMOS buffer (not Schmitt)
assign bidir_sl  = '0;    // fast slew
assign bidir_ie  = ~bidir_oe;
assign bidir_pu  = '0;
assign bidir_pd  = '0;
```

**Prerequisites.** The `gf180` container from `scripts/bootstrap_container.sh`
(repo-root level) must be running and the workspace bind-mount
(`~/eda/designs` <-> `/foss/designs`) must be in place. Runtime of
one full flow: ~35-45 min on the reference machine; Magic DRC
dominates.


> **In-container variant.** This notebook runs **inside** the `gf180`
> container (the `hpretl/iic-osic-tools:chipathon26` Docker image with
> Jupyter Lab bundled). It calls EDA tools directly with `subprocess.run`,
> not through `docker exec`. For the host-driven workflow (Jupyter on
> host, EDA tools reached via `docker exec`), use the parent directory's
> [`../03_rtl2gds_chipathon_use.ipynb`](../03_rtl2gds_chipathon_use.ipynb).
> See [`README.md`](README.md) for the in-container bootstrap recipe.


In [ ]:
import shutil
from pathlib import Path

if not Path("/.dockerenv").exists():
    raise RuntimeError(
        "This notebook is the IN-CONTAINER variant. It must run inside "
        "the gf180 container (hpretl/iic-osic-tools:chipathon26). "
        "If you have Jupyter on your host, open ../03_rtl2gds_chipathon_use.ipynb "
        "instead. See README.md for the in-container bootstrap recipe."
    )
for tool in ("librelane", "yosys", "magic", "klayout", "git", "make"):
    if shutil.which(tool) is None:
        raise RuntimeError(
            f"{tool!r} not found on PATH. Did you start the right image "
            "(hpretl/iic-osic-tools:chipathon26)?"
        )
print("OK: inside gf180 container; full toolchain on PATH")


## Step 0 — configuration

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import textwrap

# ---- flags ----
RUN_STAGE_TEMPLATE = False   # copies the in-repo workshop padring into the bind-mount (~200 KB)
RUN_CLONE_PDK      = False   # clones wafer-space GF180 PDK @ 1.8.0 (~2 min, ~500 MB)
RUN_WRITE_CORE     = False   # writes your chip_core.sv into the template
RUN_FLOW           = False   # ~35-45 min; dominated by Magic DRC

# ---- upstream references (Apache-2.0 both) ----
# The padring template is vendored next to this notebook so the flow runs
# offline. Upstream of record (for attribution): Mauricio-xx/chipathon-2026-gf180mcu-padring.
PDK_FORK_URL  = "https://github.com/wafer-space/gf180mcu.git"
PDK_FORK_TAG  = "1.8.0"

# ---- in-repo padring location: relative to this notebook (the in_container/ subdir).
#     From examples/librelane_rtl2gds_gf180/in_container/, the resources tree lives
#     three levels up (../../../resources/Integration/workshop_padring_librelane).
try:
    NB_DIR = Path(__file__).resolve().parent
except NameError:
    NB_DIR = Path.cwd().resolve()

IN_REPO_TEMPLATE = (NB_DIR / ".." / ".." / ".." / "resources" / "Integration" / "workshop_padring_librelane").resolve()
if not (IN_REPO_TEMPLATE / "librelane" / "slots" / "slot_workshop.yaml").exists():
    raise RuntimeError(
        f"In-repo padring template not found at {IN_REPO_TEMPLATE}. "
        "Launch this notebook from examples/librelane_rtl2gds_gf180/in_container/ "
        "so the relative path to resources/Integration/workshop_padring_librelane/ resolves."
    )

# ---- paths: single space (inside the container). /foss/designs is the
#     bind-mount visible to the host at ~/eda/designs.
WORKSPACE   = Path("/foss/designs")
TEMPLATE    = WORKSPACE / "chipathon_padring" / "template"
PDK_DIR     = TEMPLATE / "gf180mcu"
CORE_PATH   = TEMPLATE / "src" / "chip_core.sv"

# ---- PDK identifiers ----
PDK_NAME     = "gf180mcuD"
STD_CELL_LIB = "gf180mcu_fd_sc_mcu7t5v0"


def run_or_print(cmd, do_it, *, timeout=None, cwd=None):
    """Print the command; execute only if do_it is True.

    cmd: either a list of argv tokens (run directly) OR a bash script
    string (wrapped in `bash -lc <script>` so sak-pdk-script.sh resolves
    via the container's login shell).
    """
    if isinstance(cmd, str):
        print("$ bash -lc '<script>'")
        print(textwrap.indent(cmd, "  | "))
        args = ["bash", "-lc", cmd]
    else:
        print("$ " + " ".join(str(x) for x in cmd) + (f"   (cwd={cwd})" if cwd else ""))
        args = list(cmd)
    if not do_it:
        print("  (skipped -- flip the RUN_* flag to execute)\n")
        return None
    print("  (executing...)")
    proc = subprocess.run(args, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if proc.stdout.strip():
        print(proc.stdout[-4000:])
    if proc.returncode != 0 and proc.stderr.strip():
        print("STDERR (tail):")
        print(proc.stderr[-2000:])
    print(f"  returncode={proc.returncode}\n")
    return proc


print(f"In-repo template:  {IN_REPO_TEMPLATE}")
print(f"Workspace:         {WORKSPACE}")
print(f"Template:          {TEMPLATE}")
print(f"PDK fork:          {PDK_DIR}")
print(f"chip_core.sv path: {CORE_PATH}")


## Step 1a -- stage the in-repo workshop padring

Copies the vendored padring template
(`../../resources/Integration/workshop_padring_librelane/` from this
notebook) into `~/eda/designs/chipathon_padring/template`. The
template is small (~200 KB) and the step is idempotent.


In [ ]:
if (TEMPLATE / 'librelane' / 'slots' / 'slot_workshop.yaml').exists():
    print(f'Template already staged at {TEMPLATE}  (skipping)')
elif RUN_STAGE_TEMPLATE:
    print(f'$ shutil.copytree({IN_REPO_TEMPLATE} -> {TEMPLATE})')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    TEMPLATE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(IN_REPO_TEMPLATE, TEMPLATE, dirs_exist_ok=True)
    print(f'  staged ({sum(1 for _ in TEMPLATE.rglob("*"))} entries)\n')
else:
    print(f'$ shutil.copytree({IN_REPO_TEMPLATE} -> {TEMPLATE})')
    print('  (skipped -- flip RUN_STAGE_TEMPLATE = True to execute)\n')


## Step 1b -- clone the wafer-space GF180 PDK fork at tag 1.8.0

The padring uses I/O cells that only exist in the wafer-space fork
of the GF180MCU PDK. `make clone-pdk` inside the template would
also work; we do it explicitly here so the notebook is self-contained.


In [ ]:
if PDK_DIR.exists():
    print(f'PDK fork already cloned at {PDK_DIR}  (skipping)')
else:
    clone_pdk_cmd = ['git', 'clone', '--depth', '1', '--branch', PDK_FORK_TAG, PDK_FORK_URL, str(PDK_DIR)]
    run_or_print(clone_pdk_cmd, RUN_CLONE_PDK, timeout=600)


## Step 2 — verify the template is ready

Sanity check: the workshop slot exists, the wafer-space PDK fork is
in place, and the container is alive. If anything is missing, the
cell below tells you what to fix (most likely: run
`rtl2gds_chipathon_padring.ipynb` first, or extract the chipathon
tarball into `~/eda/designs/chipathon_padring/`).

In [ ]:
def ok(label, cond, detail=""):
    tag = "OK " if cond else "!! "
    print(f"{tag}{label}" + (f"  -- {detail}" if detail else ""))
    return cond

all_ok = True
all_ok &= ok(f"Template dir exists", TEMPLATE.exists(), str(TEMPLATE))
all_ok &= ok("slot_workshop.yaml present",
             (TEMPLATE / "librelane" / "slots" / "slot_workshop.yaml").exists())
all_ok &= ok("SLOT_WORKSHOP block in slot_defines.svh",
             "SLOT_WORKSHOP" in (TEMPLATE / "src" / "slot_defines.svh").read_text()
             if (TEMPLATE / "src" / "slot_defines.svh").exists() else False)
all_ok &= ok("wafer-space PDK fork present",
             (TEMPLATE / "gf180mcu" / "gf180mcuD").exists())
all_ok &= ok("Makefile lists 'workshop' in AVAILABLE_SLOTS",
             "workshop" in (TEMPLATE / "Makefile").read_text()
             if (TEMPLATE / "Makefile").exists() else False)
# (container running check is vacuous inside the container)

if not all_ok:
    print("\n Fix the missing items before proceeding. If the template "
          "or PDK directories are missing, flip RUN_STAGE_TEMPLATE / "
          "RUN_CLONE_PDK in Step 0 and re-run Steps 1a/1b.")


## Step 3 — write your `chip_core.sv`

The default example below implements a **visible counter** whose
state is exposed on the 20 digital config pads, plus a trivial
passthrough that keeps the 60 analog pads usable (they are left
floating inside the core so a scope probe on any `ana<N>` pin can
observe the external bondpad directly).

Replace the `YOUR CORE LOGIC` section with your own RTL. Rules:

1. Keep the port list identical to what is shown. The parameters
   `NUM_INPUT_PADS` / `NUM_BIDIR_PADS` / `NUM_ANALOG_PADS` will be
   bound by `chip_top.sv` to `{1, 20, 60}` for the workshop slot.
2. Drive every output (`input_pu`, `input_pd`, `bidir_out`,
   `bidir_oe`, ...). If you forget one, Yosys halts the flow.
3. Do not instantiate the pad cells yourself; `chip_top.sv` handles
   that.
4. If you add your own macros (SRAM, analog IP), you also need to
   patch `librelane/config.yaml` to register them under `MACROS:` --
   that goes beyond this notebook; see
   `rtl2gds_chip_top_custom.ipynb` for an example of a custom macro.

In [ ]:
CHIP_CORE_USER = '''\
// Chipathon 2026 -- your custom chip_core for the workshop padring.
// Replace the YOUR CORE LOGIC block with your own RTL.

`default_nettype none

module chip_core #(
    parameter NUM_INPUT_PADS,
    parameter NUM_BIDIR_PADS,
    parameter NUM_ANALOG_PADS
    )(
    `ifdef USE_POWER_PINS
    inout  wire VDD,
    inout  wire VSS,
    `endif

    input  wire clk,
    input  wire rst_n,

    input  wire [NUM_INPUT_PADS-1:0] input_in,
    output wire [NUM_INPUT_PADS-1:0] input_pu,
    output wire [NUM_INPUT_PADS-1:0] input_pd,

    input  wire [NUM_BIDIR_PADS-1:0] bidir_in,
    output wire [NUM_BIDIR_PADS-1:0] bidir_out,
    output wire [NUM_BIDIR_PADS-1:0] bidir_oe,
    output wire [NUM_BIDIR_PADS-1:0] bidir_cs,
    output wire [NUM_BIDIR_PADS-1:0] bidir_sl,
    output wire [NUM_BIDIR_PADS-1:0] bidir_ie,
    output wire [NUM_BIDIR_PADS-1:0] bidir_pu,
    output wire [NUM_BIDIR_PADS-1:0] bidir_pd,

    inout  wire [NUM_ANALOG_PADS-1:0] analog
);

    // ---- default-safe pad controls (keep these unless you know why you don't) ----
    assign input_pu = '0;
    assign input_pd = '0;
    assign bidir_oe = '1;    // drive outwards
    assign bidir_cs = '0;    // CMOS buffer (not Schmitt)
    assign bidir_sl = '0;    // fast slew
    assign bidir_ie = ~bidir_oe;
    assign bidir_pu = '0;
    assign bidir_pd = '0;

    // Keep synthesis from dropping un-connected inputs -- harmless.
    logic _unused;
    assign _unused = &{1'b0, bidir_in, input_in};

    // ---- YOUR CORE LOGIC STARTS HERE ----
    // Example: a 20-bit counter exposed on the 20 digital bidir pads.
    // Bits roll every ~0.5 s at 1 MHz input clock.  Replace with your RTL.
    logic [NUM_BIDIR_PADS-1:0] count;
    always_ff @(posedge clk) begin
        if (!rst_n) count <= '0;
        else        count <= count + 1;
    end
    assign bidir_out = count;
    // ---- YOUR CORE LOGIC ENDS HERE ----

    // The 60 analog pads stay untouched at the core level; bring them
    // to internal analog IP by naming each explicitly if needed, e.g.:
    //   my_opamp u_op (.vp (analog[0]), .vn (analog[1]), .out (analog[2]));

endmodule

`default_nettype wire
'''

if RUN_WRITE_CORE:
    CORE_PATH.write_text(CHIP_CORE_USER)
    print(f"Wrote {CORE_PATH}  ({len(CHIP_CORE_USER.splitlines())} lines)")
else:
    print("Preview of chip_core.sv (flip RUN_WRITE_CORE = True to commit to disk):\n")
    print(textwrap.indent(CHIP_CORE_USER, "    "))

## Step 4 — run the flow

```bash
# Inside the container:
cd /foss/designs/chipathon_padring/template
source sak-pdk-script.sh gf180mcuD gf180mcu_fd_sc_mcu7t5v0
make librelane SLOT=workshop PDK=gf180mcuD PDK_ROOT=./gf180mcu
```

Expected wall time: 35-45 min on a fast host, multiple hours if
Magic DRC is slow. `RUN_FLOW` stays `False` by default so the
cell can rehearse the command without executing it.

In [ ]:
flow_script = textwrap.dedent(f"""
    set -e
    cd {TEMPLATE}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    make librelane \\
        SLOT=workshop \\
        PDK={PDK_NAME} \\
        PDK_ROOT={TEMPLATE}/gf180mcu
""").strip()

# No timeout: full chip-top flow can run 2-3 hours on the workshop slot
# (Magic DRC + KLayout DRC dominate). Interrupt the cell from the kernel
# if you need to abort. Monitor from another shell on the host:
#   ls /foss/designs/chipathon_padring/template/librelane/runs/ | tail -1 | \
#     xargs -I{{}} ls /foss/designs/chipathon_padring/template/librelane/runs/{{}} | \
#     grep -E "^[0-9]+-" | tail
run_or_print(flow_script, RUN_FLOW, timeout=None)


## Step 5 — read `metrics.csv`

One row per metric in `final/metrics.csv`. A green chip has zeros
across every violation counter.

In [ ]:
import csv

metrics_path = TEMPLATE / "final" / "metrics.csv"
wanted = [
    "design__die__area__um2",
    "design__instance__count",
    "design__instance__count__stdcell",
    "design__instance__count__class:macro",
    "magic__drc_error__count",
    "klayout__drc_error__count",
    "design__lvs_error__count",
    "antenna__violating__nets",
    "timing__setup_vio__count",
    "timing__hold_vio__count",
    "power__total",
]

if not metrics_path.exists():
    print(f"metrics.csv not found: {metrics_path}")
    print("Did Step 3 complete?  Set RUN_FLOW = True and re-run.")
else:
    found = {}
    with metrics_path.open() as fh:
        for row in csv.reader(fh):
            if row and row[0] in wanted:
                found[row[0]] = row[1] if len(row) > 1 else ""
    for k in wanted:
        print(f"  {k:45s} {found.get(k, '(missing)')}")

## Step 6 — show the render

The final layout is at `final/gds/chip_top.gds`. LibreLane's
`KLayout.Render` step writes `final/render/chip_top.png` for
quick visual inspection.

In [ ]:
from IPython.display import Image, display

png_path = TEMPLATE / "final" / "render" / "chip_top.png"
gds_path = TEMPLATE / "final" / "gds"    / "chip_top.gds"

if png_path.exists():
    print(f"Render: {png_path}")
    display(Image(str(png_path)))
else:
    print(f"No render PNG yet at {png_path}")
    if gds_path.exists():
        print(f"GDS is there: {gds_path}")
        print(f"  open on the host:  klayout {gds_path}")

## Where to go next

1. **Multi-macro example**: See
   [`examples/04_counter_alu_multimacro/`](../04_counter_alu_multimacro/)
   for a walkthrough of hardening a counter and an ALU as independent
   macros and stitching them back into the workshop slot. That
   notebook shows the `MACROS`, `PDN_MACRO_CONNECTIONS`, and
   floorplan-halo keys you touch when you move past a single
   `chip_core`.
2. **Add custom macros** (pre-hardened blocks -- SRAM, opamp, SAR
   ADC DAC). Harden the block on its own with the Classic flow (see
   notebook 01 as a template), then reference it under `MACROS:` in
   `librelane/config.yaml`. Notebook 02 has a worked counter example.
3. **Connect analog IP** to `analog[N]` pins. The 60 analog pads in
   the workshop padring are `gf180mcu_fd_io__asig_5p0` cells; they
   tolerate the 5 V supply domain of the pad ring.
4. **Verify with cocotb** before running the heavy flow. The
   template has `make verify SLOT=workshop`.
5. **Tape out**. The workshop padring matches the shuttle's
   2935 x 2935 um tile; once DRC/LVS/timing are green, your
   `final/gds/chip_top.gds` is ready to hand in.

Clean up:

```bash
rm -rf ~/eda/designs/chipathon_padring/template/librelane/runs
docker stop gf180
```
